# Baseline Models Implementation

This notebook implements baseline models for stock price prediction:
1. LightGBM Regressor
2. Bidirectional GRU
3. Bidirectional LSTM

We'll create a robust training and testing pipeline for next-day price prediction.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import yfinance as yf
import torch
import ta
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error

## Data Preparation and Feature Engineering

In [2]:
def add_technical_indicators(df):
    """Add technical indicators to the dataframe
    
    Args:
        df (pd.DataFrame): OHLCV dataframe
        
    Returns:
        pd.DataFrame: DataFrame with technical indicators
    """
    # Price-based indicators
    df['Returns'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    
    # Volume indicators
    df['Volume_MA5'] = df['Volume'].rolling(window=5).mean()
    
    # Momentum indicators
    df['RSI'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    
    # Volatility indicators
    df['ATR'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
    
    return df

In [3]:
def create_stock_features(ticker_symbol, start_date, end_date=None):
    """Complete pipeline for fetching and processing stock data
    
    Args:
        ticker (str): Stock ticker symbol
        start_date (str): Start date in 'YYYY-MM-DD' format
        end_date (str, optional): End date in 'YYYY-MM-DD' format
        
    Returns:
        pd.DataFrame: Processed dataframe with all features
    """
    # Fetch raw data
    ticker_object = yf.Ticker(ticker_symbol)
    df = ticker_object.history(start=start_date, end=end_date)
    
    # Add technical indicators
    df = add_technical_indicators(df)
    
    # Clean data
    df = df.dropna()  # Remove any rows with missing values
    
    return df

In [4]:
def prepare_sequences(data, seq_length=20):
    """Prepare sequences for time series models
    
    Args:
        data (pd.DataFrame): Input dataframe with datetime index
        seq_length (int): Length of sequence for lookback
        
    Returns:
        tuple: (X sequences, y targets)
    """
    # Convert data to numpy, excluding the datetime index
    feature_data = data.values
    
    sequences = []
    targets = []
    
    for i in range(len(feature_data) - seq_length):
        seq = feature_data[i:(i + seq_length)]
        target = feature_data[i + seq_length][data.columns.get_loc('Close')]  # Get Close price
        sequences.append(seq)
        targets.append(target)
        
    return np.array(sequences), np.array(targets)

class StockDataset(Dataset):
    """PyTorch Dataset for stock data"""
    def __init__(self, sequences, targets):
        # Ensure data is in the correct format and type
        self.sequences = torch.FloatTensor(sequences.astype(np.float32))
        self.targets = torch.FloatTensor(targets.astype(np.float32))
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

## Model Architectures

In [5]:
class BiGRU(nn.Module):
    """Bidirectional GRU for time series prediction"""
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.2):
        super(BiGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size * 2, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

class BiLSTM(nn.Module):
    """Bidirectional LSTM for time series prediction"""
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.2):
        super(BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size * 2, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

## Training Functions

In [6]:
def train_deep_model(model, train_loader, val_loader, criterion, optimizer, 
                     device, epochs=100, early_stopping_patience=10):
    """Train deep learning models with early stopping"""
    model = model.to(device)
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for sequences, targets in train_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for sequences, targets in val_loader:
                sequences, targets = sequences.to(device), targets.to(device)
                outputs = model(sequences)
                val_loss += criterion(outputs, targets).item()
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping at epoch {epoch}')
                break
                
        if epoch % 10 == 0:
            print(f'Epoch {epoch}: Train Loss = {train_loss/len(train_loader):.4f}, '
                  f'Val Loss = {val_loss/len(val_loader):.4f}')
    
    return model

def train_lightgbm(X_train, y_train, X_val, y_val):
    """Train LightGBM model with early stopping"""
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, val_data],
        num_boost_round=1000,
        early_stopping=50,
        verbose_eval=100
    )
    
    return model

## Model Evaluation

In [7]:
def evaluate_model(model, test_loader, criterion, device, model_type='deep'):
    """Evaluate model performance"""
    if model_type == 'deep':
        model.eval()
        test_loss = 0
        predictions = []
        actuals = []
        
        with torch.no_grad():
            for sequences, targets in test_loader:
                sequences, targets = sequences.to(device), targets.to(device)
                outputs = model(sequences)
                test_loss += criterion(outputs, targets).item()
                predictions.extend(outputs.cpu().numpy())
                actuals.extend(targets.cpu().numpy())
                
        return {
            'test_loss': test_loss / len(test_loader),
            'predictions': np.array(predictions),
            'actuals': np.array(actuals)
        }
    else:  # LightGBM
        predictions = model.predict(test_loader)
        return predictions

## Example Usage

In [8]:
# Get data
ticker = '^GSPC'
df = create_stock_features(ticker, start_date='1990-01-01')


In [9]:

# Define date ranges for splits
train_end = pd.Timestamp('2010-01-01').tz_localize(df.index.tz)
val_end = pd.Timestamp('2020-01-01').tz_localize(df.index.tz)


In [10]:

# Split data by date
train_data = df[df.index < train_end]
val_data = df[(df.index >= train_end) & (df.index < val_end)]
test_data = df[df.index >= val_end]


In [11]:

# Prepare sequences for each split
X_train, y_train = prepare_sequences(train_data)
X_val, y_val = prepare_sequences(val_data)
X_test, y_test = prepare_sequences(test_data)


In [12]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)
print(df.shape)

(5004, 20, 13) (5004,)
(2496, 20, 13) (2496,)
(1251, 20, 13) (1251,)
(8811, 13)


In [13]:

# Create datasets
train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)


In [14]:

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


In [15]:

# Initialize models
input_size = X_train.shape[2]  # Number of features
hidden_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [16]:

# Initialize BiLSTM model
bilstm_model = BiLSTM(input_size=input_size, hidden_size=hidden_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=0.001)


In [17]:

# Train BiLSTM model
trained_bilstm = train_deep_model(bilstm_model, train_loader, val_loader, 
                                 criterion, optimizer, device, epochs=50)


: 

In [18]:

# Initialize and train BiGRU model
bigru_model = BiGRU(input_size=input_size, hidden_size=hidden_size)
optimizer = torch.optim.Adam(bigru_model.parameters(), lr=0.001)


In [ ]:

trained_bigru = train_deep_model(bigru_model, train_loader, val_loader, 
                                criterion, optimizer, device, epochs=50)



Epoch 0: Train Loss = 967945.7301, Val Loss = 4116554.0304
Epoch 10: Train Loss = 648274.3410, Val Loss = 3401584.6939
Epoch 20: Train Loss = 429033.9405, Val Loss = 2818369.0773
Epoch 30: Train Loss = 280885.8929, Val Loss = 2334745.3305
Epoch 40: Train Loss = 192858.0285, Val Loss = 1945787.1224


In [20]:
# Train LightGBM model
# Reshape data for LightGBM
X_train_lgb = X_train.reshape(X_train.shape[0], -1)
X_val_lgb = X_val.reshape(X_val.shape[0], -1)
X_test_lgb = X_test.reshape(X_test.shape[0], -1)


In [21]:

lgb_model = train_lightgbm(X_train_lgb, y_train, X_val_lgb, y_val)

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

: 

In [ ]:

bilstm_results = evaluate_model(trained_bilstm, test_loader, criterion, device, model_type='deep')
bigru_results = evaluate_model(trained_bigru, test_loader, criterion, device, model_type='deep')
lgb_predictions = evaluate_model(lgb_model, X_test_lgb, None, None, model_type='lgb')

# Print results
print("BiLSTM Test Loss:", bilstm_results['test_loss'])
print("BiLSTM RMSE:", np.sqrt(mean_squared_error(bilstm_results['actuals'], bilstm_results['predictions'])))
print("BiLSTM MAPE:", np.mean(np.abs((bilstm_results['actuals'] - bilstm_results['predictions']) / bilstm_results['actuals'])) * 100)

print("\nBiGRU Test Loss:", bigru_results['test_loss'])
print("BiGRU RMSE:", np.sqrt(mean_squared_error(bigru_results['actuals'], bigru_results['predictions'])))
print("BiGRU MAPE:", np.mean(np.abs((bigru_results['actuals'] - bigru_results['predictions']) / bigru_results['actuals'])) * 100)

print("\nLightGBM RMSE:", np.sqrt(mean_squared_error(y_test, lgb_predictions)))
print("LightGBM MAPE:", np.mean(np.abs((y_test - lgb_predictions) / y_test)) * 100)